# Medicaid Drug Spending — Exploratory Data Analysis
**Dataset:** CMS Medicaid Spending by Drug (2019–2023)  
**Source:** https://data.cms.gov  
**Purpose:** Explore spending patterns, cost distributions, and procurement risk signals

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#1e1e2e'
plt.rcParams['figure.facecolor'] = '#13131f'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.titlecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#444'
plt.rcParams['grid.color'] = '#333'
plt.rcParams['font.family'] = 'sans-serif'

PRIMARY = '#0066CC'
ACCENT  = '#00A86B'
WARN    = '#E63946'

# Load data
# Update this path to match where you saved the CSV
CSV_PATH = '../data/DSD_MCD_RY25_P06_V20_D23_BGM.csv'

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

## 2. Data Summary

In [ ]:
# Shape and data types
print('=== Dataset Shape ===')
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')

print('\n=== Column Data Types ===')
print(df.dtypes)

In [ ]:
# Null value summary
null_summary = pd.DataFrame({
    'Missing Values' : df.isnull().sum(),
    'Missing %'      : (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary = null_summary[null_summary['Missing Values'] > 0].sort_values('Missing %', ascending=False)

print('=== Columns with Missing Values ===')
print(null_summary.to_string())

In [ ]:
# Statistical summary for 2023 spending columns
cols_2023 = [
    'Tot_Spndng_2023',
    'Tot_Clms_2023',
    'Avg_Spnd_Per_Clm_2023',
    'Avg_Spnd_Per_Dsg_Unt_Wghtd_2023',
    'Chg_Avg_Spnd_Per_Dsg_Unt_22_23',
    'CAGR_Avg_Spnd_Per_Dsg_Unt_19_23'
]

print('=== 2023 Key Columns — Statistical Summary ===')
print(df[cols_2023].describe().round(2).to_string())

## 3. Drug Classification — Brand vs Generic

In [ ]:
# Classify each drug row as Brand or Generic
df['Drug_Type'] = df.apply(
    lambda row: 'Generic'
    if pd.isna(row['Brnd_Name']) or row['Brnd_Name'].strip() == row['Gnrc_Name'].strip()
    else 'Brand',
    axis=1
)

type_counts = df['Drug_Type'].value_counts()
print('=== Drug Type Distribution ===')
print(type_counts.to_string())

## 4. Histogram — Distribution of Average Cost Per Dosage Unit (2023)

In [ ]:
# Filter out nulls and extreme outliers for a readable chart
hist_data = df[
    df['Avg_Spnd_Per_Dsg_Unt_Wghtd_2023'].notna() &
    (df['Avg_Spnd_Per_Dsg_Unt_Wghtd_2023'] < df['Avg_Spnd_Per_Dsg_Unt_Wghtd_2023'].quantile(0.95))
]['Avg_Spnd_Per_Dsg_Unt_Wghtd_2023']

fig, ax = plt.subplots()

ax.hist(hist_data, bins=60, color=PRIMARY, edgecolor='#0a0a1a', linewidth=0.4)

ax.set_title('Distribution of Average Cost Per Dosage Unit (2023)\n(Top 5% outliers excluded)', fontsize=13, pad=12)
ax.set_xlabel('Avg Spend Per Dosage Unit (USD)')
ax.set_ylabel('Number of Drug-Manufacturer Records')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.3)

# Median line
median_val = hist_data.median()
ax.axvline(median_val, color=WARN, linestyle='--', linewidth=1.5, label=f'Median: ${median_val:,.2f}')
ax.legend()

plt.tight_layout()
plt.savefig('../assets/hist_cost_per_unit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Most drugs have a low cost per unit. A small number of high-cost drugs pull the distribution right — typical in pharmaceutical spending.')

## 5. Scatter Plot — Total Spending vs Total Claims (2023)

In [ ]:
scatter_data = df[
    df['Tot_Spndng_2023'].notna() &
    df['Tot_Clms_2023'].notna() &
    (df['Tot_Spndng_2023'] > 0) &
    (df['Tot_Clms_2023'] > 0)
].copy()

# Color by drug type
colors = scatter_data['Drug_Type'].map({'Brand': WARN, 'Generic': ACCENT})

fig, ax = plt.subplots()

ax.scatter(
    scatter_data['Tot_Clms_2023'],
    scatter_data['Tot_Spndng_2023'],
    c=colors,
    alpha=0.5,
    s=18,
    edgecolors='none'
)

ax.set_title('Total Spending vs Total Claims by Drug (2023)', fontsize=13, pad=12)
ax.set_xlabel('Total Claims')
ax.set_ylabel('Total Spending (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax.grid(alpha=0.2)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=WARN,  markersize=8, label='Brand'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=ACCENT, markersize=8, label='Generic')
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('../assets/scatter_spending_vs_claims.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: High-spending brand drugs appear at lower claim volumes — they are expensive per unit. Generic drugs serve high claim volumes at lower total cost.')

## 6. Bar Chart — Top 10 Drugs by Total Spending (2023)

In [ ]:
top10 = (
    df[df['Tot_Spndng_2023'].notna()]
    .groupby('Brnd_Name', as_index=False)['Tot_Spndng_2023']
    .sum()
    .sort_values('Tot_Spndng_2023', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    top10['Brnd_Name'],
    top10['Tot_Spndng_2023'],
    color=PRIMARY,
    edgecolor='none'
)

# Highlight the top drug
bars[0].set_color(WARN)

ax.invert_yaxis()
ax.set_title('Top 10 Drugs by Total Medicaid Spending (2023)', fontsize=13, pad=12)
ax.set_xlabel('Total Spending (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
ax.grid(axis='x', alpha=0.3)

# Value labels
for bar in bars:
    width = bar.get_width()
    ax.text(
        width * 1.01,
        bar.get_y() + bar.get_height() / 2,
        f'${width/1e9:.2f}B',
        va='center', ha='left', fontsize=9, color='white'
    )

plt.tight_layout()
plt.savefig('../assets/bar_top10_drugs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: A small number of drugs dominate total spending — a classic Pareto pattern in pharmaceutical procurement.')

## 7. Line Chart — Year-over-Year Total Medicaid Spending (2019–2023)

In [ ]:
yearly_spending = pd.DataFrame({
    'Year'          : [2019, 2020, 2021, 2022, 2023],
    'Total_Spending': [
        df['Tot_Spndng_2019'].sum(),
        df['Tot_Spndng_2020'].sum(),
        df['Tot_Spndng_2021'].sum(),
        df['Tot_Spndng_2022'].sum(),
        df['Tot_Spndng_2023'].sum()
    ]
})

fig, ax = plt.subplots()

ax.plot(
    yearly_spending['Year'],
    yearly_spending['Total_Spending'],
    color=PRIMARY,
    linewidth=2.5,
    marker='o',
    markersize=8,
    markerfacecolor=WARN
)

# Fill under line
ax.fill_between(
    yearly_spending['Year'],
    yearly_spending['Total_Spending'],
    alpha=0.15,
    color=PRIMARY
)

# Value labels
for _, row in yearly_spending.iterrows():
    ax.text(
        row['Year'],
        row['Total_Spending'] * 1.015,
        f"${row['Total_Spending']/1e9:.1f}B",
        ha='center', fontsize=9, color='white'
    )

ax.set_title('Total Medicaid Drug Spending by Year (2019–2023)', fontsize=13, pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Total Spending (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.0f}B'))
ax.set_xticks([2019, 2020, 2021, 2022, 2023])
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../assets/line_yearly_spending.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Total Medicaid drug spending shows a clear multi-year trend — review direction for budget forecasting implications.')

## 8. Brand vs Generic — Average Cost Per Claim (2023)

In [ ]:
brand_generic = (
    df[df['Avg_Spnd_Per_Clm_2023'].notna()]
    .groupby('Drug_Type')['Avg_Spnd_Per_Clm_2023']
    .mean()
    .reset_index()
    .sort_values('Avg_Spnd_Per_Clm_2023', ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 5))

bar_colors = [WARN if t == 'Brand' else ACCENT for t in brand_generic['Drug_Type']]

bars = ax.bar(
    brand_generic['Drug_Type'],
    brand_generic['Avg_Spnd_Per_Clm_2023'],
    color=bar_colors,
    width=0.4,
    edgecolor='none'
)

# Value labels
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height * 1.02,
        f'${height:,.2f}',
        ha='center', va='bottom', fontsize=11, color='white', fontweight='bold'
    )

ax.set_title('Average Cost Per Claim — Brand vs Generic (2023)', fontsize=13, pad=12)
ax.set_ylabel('Avg Spend Per Claim (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/bar_brand_vs_generic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Brand drugs cost significantly more per claim than generics. Formulary substitution to generics is a high-impact cost reduction strategy.')

## 9. High-Risk Drugs — 50%+ Spending Increase (2022 → 2023)

In [ ]:
high_risk = df[
    (df['Chg_Avg_Spnd_Per_Dsg_Unt_22_23'] > 50) &
    (df['Tot_Spndng_2023'].notna()) &
    (df['Outlier_Flag_2023'] == 0)
][['Brnd_Name', 'Gnrc_Name', 'Mftr_Name', 'Chg_Avg_Spnd_Per_Dsg_Unt_22_23', 'Tot_Spndng_2023']]\
.sort_values('Chg_Avg_Spnd_Per_Dsg_Unt_22_23', ascending=False)

print(f'=== High-Risk Drugs (50%+ Cost Increase 2022→2023) ===')
print(f'Count: {len(high_risk):,} drugs\n')
print(
    high_risk.head(15)
    .rename(columns={
        'Brnd_Name'                       : 'Brand',
        'Gnrc_Name'                       : 'Generic',
        'Mftr_Name'                       : 'Manufacturer',
        'Chg_Avg_Spnd_Per_Dsg_Unt_22_23'  : '% Change',
        'Tot_Spndng_2023'                 : 'Total Spending 2023'
    })
    .to_string(index=False)
)

## 10. Summary of Key Findings

In [ ]:
total_2023   = df['Tot_Spndng_2023'].sum()
total_2019   = df['Tot_Spndng_2019'].sum()
top_drug     = df.groupby('Brnd_Name')['Tot_Spndng_2023'].sum().idxmax()
top_spending = df.groupby('Brnd_Name')['Tot_Spndng_2023'].sum().max()
top_mftr     = df[df['Mftr_Name'] != 'Overall'].groupby('Mftr_Name')['Tot_Spndng_2023'].sum().idxmax()
brand_avg    = df[df['Drug_Type'] == 'Brand']['Avg_Spnd_Per_Clm_2023'].mean()
generic_avg  = df[df['Drug_Type'] == 'Generic']['Avg_Spnd_Per_Clm_2023'].mean()
risk_count   = len(high_risk)

print('=' * 55)
print('  KEY FINDINGS SUMMARY')
print('=' * 55)
print(f'  Total Medicaid Drug Spending (2019) : ${total_2019/1e9:>8.2f}B')
print(f'  Total Medicaid Drug Spending (2023) : ${total_2023/1e9:>8.2f}B')
print(f'  5-Year Spending Change              : ${(total_2023-total_2019)/1e9:>+8.2f}B')
print(f'  Highest Spending Drug (2023)        : {top_drug}')
print(f'  Top Drug Total Spending             : ${top_spending/1e9:>8.2f}B')
print(f'  Highest Revenue Manufacturer        : {top_mftr}')
print(f'  Avg Cost Per Claim — Brand (2023)   : ${brand_avg:>8.2f}')
print(f'  Avg Cost Per Claim — Generic (2023) : ${generic_avg:>8.2f}')
print(f'  High-Risk Drugs (50%+ increase)     : {risk_count:>8,}')
print('=' * 55)